In [1]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf
from scipy.stats import pearsonr

from rich.progress import track

import seaborn as sns

Using backend: cpu

Available hardware:

TFRT_CPU_0

## Download the data

In [2]:
generator = nl.data.MNISTGenerator(ds_size=1000)

In [6]:
ensembles = 1

for i in range(ensembles):
    network = stax.serial(
      stax.Conv(32, filter_shape=(2, 2)),
      stax.Relu(),
      stax.AvgPool(window_shape=(2, 2), strides=(2, 2)),
      stax.Flatten(),
      stax.Dense(128),
      stax.Relu(),
      stax.Dense(10)
    )

    optimizer = nl.optimizers.TraceOptimizer(
        scale_factor=1.0, subset=0.1, 
    )
#     optimizer = optax.adam(1.0)
    optimizer = optax.sgd(1.0)

    model = nl.models.NTModel(
            nt_module=network,
            optimizer=optimizer,
            seed=10,
            input_shape=(1, 28, 28, 1),
    )
    
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"traceopt_train_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"traceopt_test_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1
    )
    
    train_recorder.instantiate_recorder(
        data_set=generator.train_ds
    )
    test_recorder.instantiate_recorder(
        data_set=generator.test_ds
    )
    
    training_strategy = nl.training_strategies.SimpleTraining(
        model=model, 
        loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder],
    )
    _ = training_strategy.train_model(
            train_ds=generator.train_ds,
            test_ds=generator.test_ds, 
            epochs=500, 
            batch_size=50,
        recorders=[train_recorder, test_recorder],
        )
    
    train_recorder.dump_records()
    test_recorder.dump_records()

Epoch: 33:   7%|██▏                              | 33/500 [00:30<07:04,  1.10batch/s, accuracy=0.39]
Exception ignored in: <function _xla_gc_callback at 0x7f90f080f400>
Traceback (most recent call last):
  File "/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/jax/_src/lib/__init__.py", line 103, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 
Exception ignored in: <function UniquePtr.__del__ at 0x7f90d85f2680>
Traceback (most recent call last):
  File "/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/tensorflow/python/framework/c_api_util.py", line 70, in __del__
    def __del__(self):
KeyboardInterrupt: 

KeyboardInterrupt



In [ ]:
def load_data(file):
    with hf.File(file, "r") as db:
        data = db["loss"][:]
        
    return data

adam_data = []

for i in range(10):
    adam_data.append(
        load_data(f"adam_test_recorder_{i}.h5")
    )
    
traceopt_data = []

for i in range(10):
    traceopt_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5")
    )


x = np.linspace(1, 51, 50)

plt.errorbar(
    x,
    np.mean(traceopt_data, axis=0), 
    yerr=np.std(traceopt_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
    label="TraceOpt"
)
plt.errorbar(
    x,
    np.mean(adam_data, axis=0), 
    yerr=np.std(adam_data, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
    label="ADAM"
)

plt.xlabel("Epoch")
plt.ylabel(r"$\mathcal{L}_{test}$")
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.savefig("mpg-vs-adam.pdf")
plt.show()